# Week 6, Day 5: Symmetric matrices: the spectral theorem, covariance and energy

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
A = np.array([[2.0, 1.0, 0.0],
              [1.0, 3.0, 1.0],
              [0.0, 1.0, 2.0]])
lam, Q = np.linalg.eigh(A)
print("eigenvalues:", lam.round(3))
print("Q^T Q =\n", (Q.T @ Q).round(10))
print("Q Λ Q^T equals A:", np.allclose(Q @ np.diag(lam) @ Q.T, A))

### Exercise: Covariance and its eigenvectors

`X` holds 500 trials of noise from two neurons. Compute `Sigma` from the centred data with the formula above (not `np.cov`), then its eigenvalues `evals` and eigenvectors `evecs` sorted largest first with `eigh`. Store the direction of the top eigenvector in degrees, between 0 and 180, in `angle_deg`.

In [ ]:
rng = np.random.default_rng(1)
s1, s2, rho = 1.4, 0.8, 0.6
C = np.array([[s1 ** 2, rho * s1 * s2], [rho * s1 * s2, s2 ** 2]])
X = rng.multivariate_normal([20, 12], C, size=500)     # 500 trials, spike counts of two neurons

Xc = ...
Sigma = ...
evals, evecs = np.linalg.eigh(Sigma)
evals, evecs = ...                                     # largest first
angle_deg = ...
print(Sigma.round(3)); print("eigenvalues", evals.round(3), " long axis at", round(angle_deg, 1), "degrees")

In [ ]:
import numpy as np
def test_cov():
    "Sigma matches np.cov"
    assert np.allclose(Sigma, np.cov(X, rowvar=False))

def test_eig():
    "evals are sorted largest first and evecs has matching unit columns"
    assert np.all(np.diff(evals) <= 1e-12)
    assert np.allclose(Sigma @ evecs, evecs * evals)
    assert np.allclose(evecs.T @ evecs, np.eye(2))

def test_angle():
    "angle_deg is the direction of the top eigenvector, between 0 and 180 degrees"
    v = evecs[:, 0]
    assert 0 <= angle_deg < 180
    assert np.isclose(np.degrees(np.arctan2(v[1], v[0])) % 180, angle_deg)

test_cov()
test_eig()
test_angle()
print("All checks passed")

### Exercise: Search every direction

For each whole-degree angle from 0 to 359, form the unit vector $\mathbf{u} = (\cos\theta, \sin\theta)$ and compute the variance along it, $\mathbf{u}^\top\Sigma\mathbf{u}$, storing the 360 values in `var_u`. Find the angle of the largest value, `best_angle`, between 0 and 179, and compare the extremes with the eigenvalues.

In [ ]:
import matplotlib.pyplot as plt
thetas = np.radians(np.arange(360))
U = np.column_stack([np.cos(thetas), np.sin(thetas)])   # (360, 2), one unit vector per row
var_u = ...
best_angle = ...

plt.figure(figsize=(5, 3)); plt.plot(np.degrees(thetas), var_u)
plt.axhline(evals[0], ls="--", color="C1"); plt.axhline(evals[1], ls="--", color="C2")
plt.xlabel("direction (degrees)"); plt.ylabel("variance along u"); plt.show()
print("max", var_u.max().round(4), "at", best_angle, "degrees;  top eigenvalue", evals[0].round(4))

In [ ]:
import numpy as np
def test_var():
    "var_u has one value per angle and equals u^T Sigma u"
    assert var_u.shape == (360,)
    u = np.array([np.cos(np.radians(37)), np.sin(np.radians(37))])
    assert np.isclose(var_u[37], u @ Sigma @ u)

def test_max():
    "the largest variance is the top eigenvalue, and the smallest the bottom one"
    assert np.isclose(var_u.max(), evals[0], rtol=1e-3)
    assert np.isclose(var_u.min(), evals[1], rtol=1e-2)

def test_best():
    "best_angle is the angle (in whole degrees) of greatest variance"
    assert abs(best_angle - angle_deg) <= 1

test_var()
test_max()
test_best()
print("All checks passed")

### Exercise: Energy only goes downhill

Write `energy(x, W, b)`. Then simulate both networks below with Euler steps (τ = 1, dt = 0.01, 600 steps) and record the energy at every step in `E_sym` and `E_non` (use each network's own W in the formula). Which one goes only downhill?

In [ ]:
def energy(x, W, b):
    return ...

rng = np.random.default_rng(5)
M = rng.normal(size=(5, 5))
W_sym = 0.25 * (M + M.T)                   # symmetric
W_non = 0.5 * (M - M.T) + 0.1 * np.eye(5)  # mostly antisymmetric: it rotates activity
b = rng.normal(size=5)
dt, n_steps = 0.01, 600

def run(W):
    x = rng.normal(size=5) * 2
    Es = [energy(x, W, b)]
    for _ in range(n_steps):
        x = ...
        Es.append(energy(x, W, b))
    return np.array(Es)

print("largest eigenvalue of W_sym:", np.linalg.eigvalsh(W_sym).max().round(3))
E_sym, E_non = run(W_sym), run(W_non)
plt.figure(figsize=(5, 3)); plt.plot(E_sym, label="symmetric W"); plt.plot(E_non, label="non-symmetric W")
plt.xlabel("step"); plt.ylabel("E"); plt.legend(frameon=False); plt.show()

In [ ]:
import numpy as np
def test_energy_fn():
    "energy computes 0.5 x^T (I - W) x - b^T x"
    Wt = np.array([[0.2, 0.1], [0.1, 0.3]]); bt = np.array([1.0, -1.0]); xt = np.array([2.0, 1.0])
    assert np.isclose(energy(xt, Wt, bt), 0.5 * xt @ (np.eye(2) - Wt) @ xt - bt @ xt)

def test_monotone():
    "E_sym never increases along the symmetric network's trajectory"
    assert len(E_sym) == n_steps + 1
    assert np.all(np.diff(E_sym) <= 1e-9)

def test_nonsym():
    "for the non-symmetric network, the same formula is not always decreasing"
    assert np.any(np.diff(E_non) > 1e-6)

test_energy_fn()
test_monotone()
test_nonsym()
print("All checks passed")